In [ ]:
import pandas as pd
import signal
import sys

from scapy.all import sniff, IP, TCP
from datetime import datetime

packet_buffer = []

def process_captured_packet(packet):

    if packet.haslayer(IP) and packet.haslayer(TCP):

        packet_data = {

            "Time": datetime.now().strftime("%H:%M:%S.%f")[:-3],
            "Src_IP": packet[IP].src,
            "Dst_IP": packet[IP].dst,
            "S_Port": packet[TCP].sport,
            "D_Port": packet[TCP].dport,
            "Flags": int(packet[TCP].flags),
            "Len": len(packet),
            "Win": packet[TCP].window,
            "TTL": packet[IP].ttl,
            "Seq": packet[TCP].seq,
            "Ack": packet[TCP].ack,
            "Payload_Size": len(packet[TCP].payload)
        }

        packet_buffer.append(packet_data)
        print(f"Catturati: {len(packet_buffer)} pacchetti", end="\r")

def save_and_exit(signal_number, frame):

    print(f"\n[INFO] Generazione dataset in corso...")

    if packet_buffer:

        dataframe = pd.DataFrame(packet_buffer)

        dataframe['IAT'] = pd.to_datetime(

            dataframe['Time'], format='%H:%M:%S.%f'

        ).diff().dt.total_seconds().fillna(0)

        output_filename = "dataset_tesi_integrale.csv"
        dataframe.to_csv(output_filename, index=False)
        print(f"[OK] Dataset salvato: {output_filename} ({len(dataframe)} righe)")
        print("\nAnteprima del dataset (prime 5 righe):")
        print(dataframe.head())
        
    sys.exit(0)

signal.signal(signal.SIGINT, save_and_exit)

print("Avvio sniffer per dataset tesi integrale")

sniff(iface="eth0", prn=process_captured_packet, store=0)